In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
drivers_df =spark.read.parquet(f"{processed_folder_path}/drivers").withColumnRenamed("name", "driver_name").withColumnRenamed("number", "driver_number").withColumnRenamed("nationality", "driver_nationality")
constructors_df =spark.read.parquet(f"{processed_folder_path}/constructors").withColumnRenamed("name", "team")
circuits_df =spark.read.parquet(f"{processed_folder_path}/circuits").filter("circuit_id < 70").withColumnRenamed("name", "circuit_name").withColumnRenamed("location", "circuit_location")
races_df =spark.read.parquet(f"{processed_folder_path}/races").withColumnRenamed("name", "race_name").withColumnRenamed("race_timestamp", "race_date")
results_df =spark.read.parquet(f"{processed_folder_path}/results").withColumnRenamed("time", "race_time")


In [0]:
race_circuits_df = circuits_df.join(races_df, circuits_df.circuit_id == races_df.circuit_id, "inner").select(circuits_df.circuit_name, circuits_df.circuit_location, circuits_df.country, races_df.race_name, races_df.race_id, races_df.race_year)

In [0]:
race_result_df = results_df.join(race_circuits_df, results_df.race_id == race_circuits_df.race_id) \
                            .join(drivers_df, results_df.driver_id == drivers_df.driver_id) \
                            .join(constructors_df, results_df.constructor_id == constructors_df.constructor_id)


In [0]:
from pyspark.sql.functions import current_timestamp
final_df = race_result_df.select("race_year","race_name", "team", "driver_name", "driver_nationality", "race_time", "position", "fastest_lap", "fastest_lap_time", "fastest_lap_speed", "grid", "laps", "points") \
                            .withColumn("created_date", current_timestamp())
        
display(final_df.filter("race_year == 2020 and race_name == 'Abu Dhabi Grand Prix'").orderBy(final_df.points.desc()))


In [0]:
final_df.write.mode("overwrite").format("parquet").saveAsTable("f1_presentation.race_results")

In [0]:
%sql
select * from f1_presentation.race_results